# 03 — Beam Chromaticity Fitting

Synthesise beam widths at multiple frequencies and fit all three chromatic models:

| Model | Formula |
|---|---|
| `powerlaw` | σ(ν) = σ₀ · (ν/ν₀)^α |
| `powerlaw_ripple` | σ(ν) = σ₀ · (ν/ν₀)^α · [1 + A · sin(2π·ν/P·ν₀ + φ)] |
| `lambda_over_D` | σ(ν) = k · λ(ν) / D |

**No external data needed.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from beamlab21.metrics import fit_beam_chromaticity

## 1 · Synthesise chromatic beam widths

### 1a · Pure power-law

In [ ]:
rng = np.random.default_rng(7)

freq_mhz  = np.linspace(300, 1000, 50)
nu0       = 600.0          # reference frequency [MHz]
SIGMA0    = 9.0            # beam width at ν₀ [deg]
ALPHA     = -0.98          # spectral index (≈ −1 for diffraction-limited dish)
NOISE_STD = 0.04           # deg

sigx_pl = SIGMA0 * (freq_mhz / nu0) ** ALPHA + rng.normal(0, NOISE_STD, len(freq_mhz))
sigy_pl = SIGMA0 * 0.92 * (freq_mhz / nu0) ** ALPHA + rng.normal(0, NOISE_STD, len(freq_mhz))

print(f"Frequencies : {freq_mhz[0]:.0f}–{freq_mhz[-1]:.0f} MHz  ({len(freq_mhz)} channels)")

### 1b · Power-law + sinusoidal ripple (standing-wave signature)

In [ ]:
A_ripple = 0.04    # fractional ripple amplitude
P_mhz    = 80.0    # ripple period [MHz]
phi_rip  = 0.8     # phase [rad]

nu_norm  = freq_mhz / nu0
ripple   = 1.0 + A_ripple * np.sin(2 * np.pi * nu_norm / (P_mhz / nu0) + phi_rip)

sigx_rip = SIGMA0 * (nu_norm) ** ALPHA * ripple + rng.normal(0, NOISE_STD, len(freq_mhz))
sigy_rip = SIGMA0 * 0.92 * (nu_norm) ** ALPHA * ripple + rng.normal(0, NOISE_STD, len(freq_mhz))

## 2 · Fit: `powerlaw`

In [ ]:
pl = fit_beam_chromaticity(freq_mhz, sigx_pl, sigy_pl,
                           nu0_mhz=nu0, model="powerlaw")

print("=== powerlaw fit ===")
print(f"σ₀_x  = {pl['sigma0_x']:.4f}°   (true {SIGMA0:.2f}°)")
print(f"α_x   = {pl['alpha_x']:.4f}     (true {ALPHA:.2f})")
print(f"σ₀_y  = {pl['sigma0_y']:.4f}°")
print(f"α_y   = {pl['alpha_y']:.4f}")

## 3 · Fit: `powerlaw_ripple`

In [ ]:
rip = fit_beam_chromaticity(freq_mhz, sigx_rip, sigy_rip,
                            nu0_mhz=nu0, model="powerlaw_ripple")

print("=== powerlaw_ripple fit ===")
print(f"σ₀_x  = {rip['sigma0_x']:.4f}°     (true {SIGMA0:.2f}°)")
print(f"α_x   = {rip['alpha_x']:.4f}       (true {ALPHA:.2f})")
print(f"A_x   = {rip['A_x']:.4f}           (true {A_ripple:.4f})")
print(f"P_x   = {rip['P_mhz_x']:.2f} MHz   (true {P_mhz:.1f} MHz)")
print(f"φ_x   = {rip['phi_x']:.4f} rad     (true {phi_rip:.4f} rad)")

## 4 · Fit: `lambda_over_D`

In [ ]:
DISH_M = 6.0

lod = fit_beam_chromaticity(freq_mhz, sigx_pl, sigy_pl,
                            model="lambda_over_D", dish_diameter_m=DISH_M)

print("=== lambda_over_D fit ===")
print(f"k_x = {lod['k_x']:.4f}   (illumination coefficient x)")
print(f"k_y = {lod['k_y']:.4f}   (illumination coefficient y)")
print(f"D   = {lod['dish_diameter_m']:.1f} m")

## 5 · Compare all three fits

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, (sigx_data, sigy_data, fit_pl, fit_rip, title) in zip(
    axes,
    [
        (sigx_pl,  sigy_pl,  pl,  None, "Pure power-law data"),
        (sigx_rip, sigy_rip, None, rip, "Power-law + ripple data"),
    ]
):
    ax.scatter(freq_mhz, sigx_data, s=10, label="σ_x data", color="steelblue", zorder=3)
    ax.scatter(freq_mhz, sigy_data, s=10, label="σ_y data", color="salmon",    zorder=3)

    f_fit = fit_pl or fit_rip
    ax.plot(f_fit["freq_mhz"], f_fit["sigma_fit_x"], "b-",   linewidth=1.8,
            label=f"fit x ({f_fit['model']})")
    ax.plot(f_fit["freq_mhz"], f_fit["sigma_fit_y"], "r--",  linewidth=1.8,
            label="fit y")

    # λ/D model on top
    ax.plot(lod["freq_mhz"], lod["sigma_fit_x"], "k:", linewidth=1.5, label="λ/D fit")

    ax.set_xlabel("Frequency [MHz]")
    ax.set_ylabel("σ [deg]")
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6 · Ripple recovery: residual spectrum

In [ ]:
# Remove power-law trend from ripple data and check FFT
pl_on_rip = fit_beam_chromaticity(freq_mhz, sigx_rip, sigy_rip,
                                  nu0_mhz=nu0, model="powerlaw")
resid = sigx_rip - pl_on_rip["sigma_fit_x"]

fft_amp = np.abs(np.fft.rfft(resid))
df      = (freq_mhz[1] - freq_mhz[0])          # channel spacing [MHz]
freqs_fft = np.fft.rfftfreq(len(resid), d=df)   # cycles / MHz
periods   = np.where(freqs_fft > 0, 1.0 / freqs_fft, np.inf)  # MHz

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

axes[0].plot(freq_mhz, resid)
axes[0].axhline(0, color="k", linewidth=0.7)
axes[0].set_xlabel("Frequency [MHz]")
axes[0].set_ylabel("σ_x residual [deg]")
axes[0].set_title("Residual after removing power-law")

axes[1].plot(periods[1:], fft_amp[1:], marker="o", markersize=3)
axes[1].axvline(P_mhz, color="r", linestyle="--", label=f"True period {P_mhz} MHz")
axes[1].set_xlabel("Period [MHz]")
axes[1].set_ylabel("|FFT|")
axes[1].set_title("FFT of residual (ripple detection)")
axes[1].legend()
axes[1].set_xlim(0, 400)

plt.tight_layout()
plt.show()